In [1]:
from pathlib import Path
from PIL import Image

# ===== folders =====
input_folder = Path("temp_logos")
output_folder = Path("converted_logos")

THRESHOLD = 245
SUPPORTED_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


def is_supported_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in SUPPORTED_EXTS


def convert_to_black_white(img: Image.Image, threshold: int = THRESHOLD) -> Image.Image:
    # ORIGINAL VERSION — unchanged
    has_alpha = "A" in img.getbands()

    if has_alpha:
        rgba = img.convert("RGBA")
        out = Image.new("RGBA", rgba.size)
        src = rgba.load()
        dst = out.load()

        for y in range(rgba.height):
            for x in range(rgba.width):
                r, g, b, a = src[x, y]

                if a == 0:
                    dst[x, y] = (255, 255, 255, 0)
                elif r >= threshold and g >= threshold and b >= threshold:
                    dst[x, y] = (255, 255, 255, a)
                else:
                    dst[x, y] = (0, 0, 0, a)

        return out

    rgb = img.convert("RGB")
    out = Image.new("RGB", rgb.size)
    src = rgb.load()
    dst = out.load()

    for y in range(rgb.height):
        for x in range(rgb.width):
            r, g, b = src[x, y]
            if r >= threshold and g >= threshold and b >= threshold:
                dst[x, y] = (255, 255, 255)
            else:
                dst[x, y] = (0, 0, 0)

    return out


def convert_lbs(img: Image.Image, threshold: int = THRESHOLD) -> Image.Image:
    # ONLY FOR LBS
    has_alpha = "A" in img.getbands()

    if has_alpha:
        rgba = img.convert("RGBA")
        out = Image.new("RGBA", rgba.size)
        src = rgba.load()
        dst = out.load()

        for y in range(rgba.height):
            for x in range(rgba.width):
                r, g, b, a = src[x, y]

                is_white = (r >= threshold and g >= threshold and b >= threshold)
                is_red = (r > 120 and r > g * 1.4 and r > b * 1.4)

                if a == 0:
                    dst[x, y] = (255, 255, 255, 0)
                elif is_white or is_red:
                    dst[x, y] = (255, 255, 255, a)
                else:
                    dst[x, y] = (0, 0, 0, a)

        return out

    rgb = img.convert("RGB")
    out = Image.new("RGB", rgb.size)
    src = rgb.load()
    dst = out.load()

    for y in range(rgb.height):
        for x in range(rgb.width):
            r, g, b = src[x, y]

            is_white = (r >= threshold and g >= threshold and b >= threshold)
            is_red = (r > 120 and r > g * 1.4 and r > b * 1.4)

            if is_white or is_red:
                dst[x, y] = (255, 255, 255)
            else:
                dst[x, y] = (0, 0, 0)

    return out


def pad_to_square_white(img: Image.Image) -> Image.Image:
    """
    Pad image to a square using white background.
    Keeps original content centered.
    """
    if img.mode != "RGB":
        img = img.convert("RGB")

    w, h = img.size
    side = max(w, h)

    square = Image.new("RGB", (side, side), (255, 255, 255))
    x = (side - w) // 2
    y = (side - h) // 2
    square.paste(img, (x, y))

    return square


output_folder.mkdir(parents=True, exist_ok=True)

image_files = [p for p in input_folder.iterdir() if is_supported_image(p)]
print(f"Found {len(image_files)} image(s) in {input_folder}")

for img_path in image_files:
    try:
        with Image.open(img_path) as img:
            if "lbs" in img_path.stem.lower():
                converted = convert_lbs(img)
                print(f"LBS rule: {img_path.name}")
            else:
                converted = convert_to_black_white(img)
                print(f"Original rule: {img_path.name}")

            converted = pad_to_square_white(converted)

            out_path = output_folder / f"{img_path.stem}_bw.png"
            converted.save(out_path)
            print(f"Saved: {out_path}")

    except Exception as e:
        print(f"Failed: {img_path.name} -> {e}")

Found 14 image(s) in temp_logos
Original rule: ARUP.png
Saved: converted_logos\ARUP_bw.png
Original rule: BMW.png
Saved: converted_logos\BMW_bw.png
Original rule: bugatti_rimac.png
Saved: converted_logos\bugatti_rimac_bw.png
Original rule: danfoss.png
Saved: converted_logos\danfoss_bw.png
Original rule: ETH.jpg
Saved: converted_logos\ETH_bw.png
Original rule: Exeter.png
Saved: converted_logos\Exeter_bw.png
Original rule: KPMG.png
Saved: converted_logos\KPMG_bw.png


c:\Users\YuR\AppData\Local\anaconda3\lib\site-packages\PIL\Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


LBS rule: LBS.png
Saved: converted_logos\LBS_bw.png
Original rule: LMU.png
Saved: converted_logos\LMU_bw.png
Original rule: Mckinsey.png
Saved: converted_logos\Mckinsey_bw.png
Original rule: RPTU.png
Saved: converted_logos\RPTU_bw.png
Original rule: TU Delft.png
Saved: converted_logos\TU Delft_bw.png
Original rule: TUMunich.png
Saved: converted_logos\TUMunich_bw.png
Original rule: Vienna.png
Saved: converted_logos\Vienna_bw.png
